In [1]:
import numpy as np
from scipy.signal import place_poles
import numpy.typing as npt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, NonlinearConstraint
N = 6
observable = False

def init_system(N:int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    A = np.eye(N, k=-1)
    A[:, -1] = np.random.randint(-N*2, -1, N)
    B = np.random.randint(-N, N, N).reshape(N,1)
    C = np.eye(1,N, k=N-1)
    return A, B, C

def get_observ_params(A: np.ndarray, C: np.ndarray, N :int) -> np.ndarray:
    desired_poles = np.random.choice(np.arange(-15, -6, 1), N, replace=False)
    res = place_poles(A.T, C.T, desired_poles)
    return res.gain_matrix

# def f(N) -> npt.NDArray:
#         return np.round(
            
#             np.random.randint(-N*3, N*3,N)
#             + np.random.random(N),
#             3,
#         ).reshape(1,N)
def f(N) -> npt.NDArray:
        l_true = -np.random.exponential(2, N-2)
        g_coeffs = np.random.normal(0, 3, N-2) 
        t = np.arange(N-1)
        basis_matrix = np.array([lt**N for lt in l_true]).T
        f_true = basis_matrix @ g_coeffs
        f_noisy = f_true + np.random.normal(0, 1, N)
        return f_noisy.reshape(1,N)

A, B, C = init_system(N)
while not observable:
    Obsv = np.vstack([ C @ np.linalg.matrix_power(A, i) for i in range(N)])
    if np.linalg.matrix_rank(Obsv) < A.shape[0]:
        print(Obsv)
        print(np.linalg.matrix_rank(Obsv))
        A, B, C = init_system(N)
    else:
        observable = True

stable = False
F = f(N)
k_sigma = 1.5
while not stable:
    A_cl = A - k_sigma * B @ F
    if np.all(np.linalg.eigvals(A_cl) < 0):
        stable = True
    else:
        # print(F)
        F = f(N)    
L = get_observ_params(A, C, N)

In [2]:
A, B, C, F, L, k_sigma, N = (np.array([[ 0.,  0.,  0.,  0.,  0., -2.],
        [ 1.,  0.,  0.,  0.,  0., -4.],
        [ 0.,  1.,  0.,  0.,  0., -6.],
        [ 0.,  0.,  1.,  0.,  0., -8.],
        [ 0.,  0.,  0.,  1.,  0., -6.],
        [ 0.,  0.,  0.,  0.,  1., -3.]]),
 np.array([[ 4],
        [ 4],
        [ 2],
        [-2],
        [ 5],
        [-1]]),
 np.array([[0., 0., 0., 0., 0., 1.]]),
 np.array([[ 13.422,  12.56 ,  -1.103, -14.334,   7.709, -16.452]]),
 np.array([[1.03487800e+06, 6.34923997e+05, 1.60437999e+05, 2.13719999e+04,
         1.57900000e+03, 5.89999999e+01]]),
 1.5,
 6)

In [3]:
def wandermond_lambdas_matrix(lambdas: npt.NDArray, n: int) -> npt.NDArray:
    return np.vstack([lambdas**i for i in range(n)]).T

def wandermond_matrix(gammas: npt.NDArray, lambdas: npt.NDArray, n: int) -> npt.NDArray:
    return np.vstack([gammas * lambdas**i for i in range(n)]).T

def pairwise_cosines(x, K, N):
    if K == 1: return 0
    gammas, lambdas = x[:K], x[K:]
    V = wandermond_lambdas_matrix( lambdas, N)      # shape (K, N)
    norms = np.linalg.norm(V, axis=1, keepdims=True)
    if np.any(norms < 1e-12):
        # возвращаем большие значения, чтобы constraint точно нарушался
        m = K * (K - 1) // 2
        return np.full(m, 1e6)
    Vn = V / norms
    G = Vn @ Vn.T
    vals = []
    for i in range(K):
        for j in range(i + 1, K):
            vals.append(G[i, j])   # cos(v_i, v_j)
    return np.array(vals)


def approximation_elemnet(
    gammas: npt.NDArray, lambdas: npt.NDArray, n: int
) -> npt.NDArray:
    return np.sum(wandermond_matrix(gammas, lambdas, n), axis=0)


def functional_J(
    f: npt.NDArray, gammas: npt.NDArray, lambdas: npt.NDArray, n: int
) -> float:
    diff = (f - approximation_elemnet(gammas, lambdas, n)).flatten()
    return diff @ diff

def x0(K: int, lambdas: list[float] | None, gammas: list[float] | None) -> npt.NDArray:
    if lambdas is not None and gammas is not None:
        return gammas + [1.0] + lambdas + [- 0.1]    
    k_range = np.arange(1,K+1, 1)
    lambdas = -0.1*k_range
    gammas= k_range
    return np.concatenate([gammas, lambdas])

def approximation_F(F: npt.NDArray, K:int, N: int, lambdas: list[float] = None, gammas: list[float] = None) -> npt.NDArray:
    alpha = 0.9
    def objective(x: npt.NDArray) -> float: 
        assert x.size == 2 * K
        gammas, lambdas = x[: K], x[K :]

        for i in range(K):
            for j in range(i + 1, K):
                distance = np.abs(lambdas[i] - lambdas[j])
                if distance < 1e-6:
                    return 1e40
        return functional_J(F, gammas, lambdas, N) +  alpha
    b_gamma = [-1e9, 1e9]
    b_lambda = [-1e9, -1e-3]
    bounds = [b_gamma] * K + [b_lambda] * K
    
    m = 1 if K == 1 else K * (K - 1) // 2
    ortho_constr = NonlinearConstraint(
        fun=lambda x: pairwise_cosines(x, K, N),
        lb=-alpha * np.ones(m),
        ub= alpha * np.ones(m),
    )
    # x0 = np.concatenate([np.random.normal(0, 2, K), -np.random.exponential( 1, K)])
    print("xo", x0(K, lambdas, gammas))
    solution = minimize(
        objective, 
        x0=x0(K, lambdas, gammas), 
        bounds=bounds,
        # constraints=[ortho_constr],
        # method="SLSQP"
        method='Nelder-Mead'
    )
    print("cosines", pairwise_cosines(solution.x, K, N))
    return solution.x[:K], solution.x[K:]



In [4]:
def run_system(A: np.ndarray, B: np.ndarray, F: np.ndarray, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        u = -k_sigma * F @ x
        x_dot = A @ x + B @ u
        return x_dot.flatten()
    
    x0 = np.ones(N)
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    sigma_ideal = (F @ sol.y).flatten()
    u_ideal = -k_sigma * sigma_ideal.flatten()
    return  u_ideal, sigma_ideal

u_ideal, sigma_ideal = run_system(A, B, F, N, 0.1, 100)
# u_ideal, sigma_ideal

In [5]:
def run_full_observe(A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, F: np.ndarray, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        x_true, x_observe = x[:N], x[N:]
        u = -k_sigma * F @ x_observe
        dx_true = A @ x_true + B @ u
        dx_observe =  A @ x_observe + B @ u + (L * (C @ x_true - C @ x_observe)).flatten()
        return np.concatenate([dx_true, dx_observe]).flatten()
    
    x0 = np.concatenate([np.ones(N), np.zeros(N)])
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    x_true, x_observe = sol.y[:N], sol.y[N:] 
    sigma_full = (F @ x_observe).flatten()   
    u_full = -k_sigma * sigma_full.flatten()
    
    return  u_full, sigma_full

u_full, sigma_full = run_full_observe(A, B, C, L, F, N, 0.1, 100)
# u_full, sigma_full

In [6]:
g_lambdas, g_gammas = None, None


def run_func_observe(A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, F: np.ndarray,K: int, N: int, dt:float=0.1, T: int =100) -> np.ndarray:
    global g_lambdas, g_gammas
    gammas, lambdas = approximation_F(F, K, N, g_lambdas, g_gammas)
    g_lambdas, g_gammas = lambdas.tolist(), gammas.tolist()
    gamma_0 = (F[C==1] - gammas@(lambdas**(N-1))).item()
    print(f"MinJ: {functional_J(F, gammas, lambdas, N)}")
    print(lambdas, gammas, gamma_0)
    LAMBDAS = wandermond_lambdas_matrix(lambdas, N)
    H = (LAMBDAS@B).flatten()
    # print(H)
    G = np.concatenate([-pow_l @ (np.eye(N) * l - A)@C.T for pow_l , l in zip(LAMBDAS, lambdas)])
    # print(G)
    def system_dynamics(t: float, x: np.ndarray) -> np.ndarray:
        x_true, sigma = x[:N], x[N:]
        sigma_observe = gammas@sigma +gamma_0*C@x_true
        u = -k_sigma * sigma_observe
        # print(A @ x_true)
        # print(B * u)
        dx_true = A @ x_true + (B * u).flatten()
        # print(lambdas*  sigma_observe)
        # print(H*u)
        # print(G*(C @ x_true))
        dsigma_observe = lambdas* sigma + H*u + G*(C @ x_true)
        # print(lambdas @ (F @ x_observe))
        # print(sigma)
        # print(dx_true)
        # print(dsigma_observe)
        return np.concatenate([dx_true, dsigma_observe]).flatten()
    
    x0 = np.concatenate([np.ones(N), np.zeros(K)])
    t = np.arange(0, T, dt)
    sol = solve_ivp(system_dynamics, [0, t[-1]], x0, t_eval=t)
    x_true, sigma_observe = sol.y[:N], sol.y[N:]  
    y = (C @ x_true).flatten()  
    sigma_func = gammas@sigma_observe +gamma_0*y
    u_func = -k_sigma * sigma_func
    
    return  u_func, sigma_func

func_observer = {k: run_func_observe(A, B, C, L, F, k, N, 0.1, 100) for k in range(1, N)}

xo [ 1.  -0.1]
cosines 0
MinJ: 694.8679618964081
[-0.001] [13.40944452] -16.451999999999988
xo [13.409444522955903, 1.0, -0.001, -0.1]
cosines [0.99969638]
MinJ: 536.4005736042383
[-0.0256396 -0.001    ] [-510.66654902  524.08856837] -16.452005658399838
xo [-510.6665490172592, 524.08856837129, 1.0, -0.025639600406573225, -0.001, -0.1]
cosines [0.99796696 0.44631434 0.41556402]
MinJ: 60.11855280114635
[-0.25681484 -0.19629398 -1.0764409 ] [-407.77869433  410.87286641   10.39327504] -1.7666411103372948
xo [-407.77869432749924, 410.8728664135266, 10.393275038986179, 1.0, -0.2568148403081051, -0.19629397636520662, -1.0764408996325794, -0.1]
cosines [0.99946967 0.46582301 0.99878496 0.44964627 0.99985981 0.44156683]
MinJ: 60.0557967239428
[-0.25083498 -0.22006867 -1.05518635 -0.20407647] [-783.73297227  702.23207526   11.45248764   83.51762275] -1.8570313664856553
xo [-783.7329722685814, 702.2320752633814, 11.452487639959468, 83.5176227495139, 1.0, -0.25083498043773483, -0.22006866946199344

In [8]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

dash_styles = ['dash', 'dot', 'dashdot', 'longdash', 'longdashdot', 'solid']

fig = go.Figure(
    layout=dict(
        legend=dict(
            yanchor="top",
            y=-0.2,
            xanchor="left",
            x=0.01,
            orientation="h",
        )
    )
)
fig.update_layout(
    xaxis_title="Время, t",
    yaxis_title="Управление, u",
    plot_bgcolor="white",
)
fig.update_xaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.update_yaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
t=np.arange(0, 10, 0.1)
fig.add_trace(go.Scatter(x=t, y=u_ideal, name='Полный наблюдатель'),)
# fig.add_trace(go.Scatter( y=u_full, name='u_full'), row=1, col=1)
for k, (u_func, sigma_func) in func_observer.items():
    dash_с = dash_styles[k % len(dash_styles)]
    fig.add_trace(go.Scatter(x=t, y=u_func, name=f'Наблюдатель при k={k}', line=dict(dash=dash_с)), )

# fig.add_trace(go.Scatter( y=sigma_ideal, name='sigma_ideal'), row=2, col=1)
# fig.add_trace(go.Scatter( y=sigma_full, name='sigma_full'), row=2, col=1)
# for k, (u_func, sigma_func) in func_observer.items():
#     dash_с = dash_styles[k % len(dash_styles)]
#     fig.add_trace(go.Scatter(y=sigma_func, name=f'sigma_func_{k}', line=dict(dash=dash_с)), row=2, col=1)
fig.update_layout(height=600, width=1200)
fig.show()

Основная идея эксперимента состояла в следующем. Вектор $F$, задающий наблюдаемый функционал состояния, аппроксимировался линейной комбинацией векторов специального вида
$$
F(\lambda_i)=(1,\lambda_i,\lambda_i^2,\ldots,\lambda_i^{n-1}), \qquad \lambda_i<0.
$$
Для каждого числа (k) строилось приближение
$$
F\approx \sum_{i=1}^k \gamma_i F(\lambda_i).
$$
Затем для каждого $F_i=\sum_{i=1}^k \gamma_i F(\lambda_i)$ строился скалярный функциональный наблюдатель

Было рассмотрено несколько способов выбора параметров аппроксимации. 

В первом варианте параметры $\lambda_i$ фиксировались заранее: $\lambda_i=-0.1i, i=1,\ldots,k$. При таких значениях коэффициенты $\gamma_i$ находились из задачи наименьших квадратов:
$$
\gamma=F\Lambda_k^\top(\Lambda_k\Lambda_k^\top)^{-1}.
$$

Во втором варианте использовался другой фиксированный набор параметров: $\lambda_i=-0.2i$, $i=1,\ldots,k$. 

Этот эксперимент показывает, что выбор узлов $\lambda_i$ существенно влияет на качество аппроксимации, однако общая тенденция уменьшения ошибки при росте $k$ сохраняется.

В третьем варианте параметры $\lambda_i$ и $\gamma_i$ подбирались одновременно с помощью численной нелинейной оптимизации. Для каждого $k=1,\ldots,4$ решалась задача минимизации
$$
\min_{\lambda_i<0,\gamma_i}
\left|F-\sum_{i=1}^k\gamma_iF(\lambda_i)\right|_2^2.
$$


После нахождения параметров $\lambda_i$, $\gamma_i$ для каждого $k$ моделировалась замкнутая система с управлением
$$
\tilde u_k=-1.5\tilde\sigma_k.
$$
Тем самым сравнивались не только ошибки аппроксимации вектора $F$, но и влияние этой ошибки на фактическое поведение замкнутой системы: траектории управления, оценки функционала и динамику состояния.